In [1]:
import sys
import os

# Add the src/ folder to the system path
sys.path.append(os.path.abspath(os.path.join('src')))

# Now you can import the module
from py_utils import *


BamPath = "/Users/liqun/scMutraceProject/Data/GSE99915/data/bam/ExcludeBG_hf1.d0.sort.rmdupicard.bam"
referencePath = "/Users/liqun/reference/mm10/mm10_nochr.fa"
outVcfPath = "/Users/liqun/scMutraceProject/Data/GSE99915/"
config_posList = ["1", 3274000, 3274000]
min_bq = 20
min_mq = 60
dp_check = True
alt_check = True
cell_check = True
nm_check = True
min_cov = 1
min_alt = 1
min_cell = 1
max_nm = 5
cbtag = "CB"
target_extractsnp_fromBAM(BamPath, referencePath, outVcfPath, config_posList, min_bq, min_mq, dp_check, alt_check, cell_check, nm_check, min_cov, min_alt, min_cell, max_nm, cbtag, 10, "hf1_d0")

In [8]:
def check_ifMut_oneCell(cellInfo, refBase):
    basicBase = ["A", "T", "C", "G", "D", "I"]
    countDic = {base: 0 for base in basicBase}
    check_base = list(set(basicBase) - {refBase})
    cellInfoList = list(map(int, cellInfo.split("|")))
    for base, count in zip(basicBase, cellInfoList):
        countDic[base] += count
    numCheck = sum(countDic[base] for base in check_base)
    return numCheck >= 1

check_ifMut_oneCell("3|1|1|0|1|1", "A")

True

In [13]:
import pandas as pd
from io import StringIO

data = '''
12_70902150    hf1.d0_TTCGGTCGTGGTACAG    WT
12_70902150    hf1.d0_CATATGGGTATATGGA    Mut
12_70902150    hf1.d0_AAATGCCGTCAAACTC    WT
12_70949151    hf1.d0_AGGCCACCATTAGGCT    WT
12_70949151    hf1.d6_CTGATCCAGGTGGGTT    WT
12_70949151    hf1.d6_ACGGGCTGTACCTACA    Mut
12_70949151    hf1.d6_TGCGGGTGTCCCGACA    WT
12_70949151    hf1.d6_GTACTCCTCAGCGACC    WT
12_70949151    hf1.d6_CGCTATCTCGAATGCT    WT
12_70982708    hf1.d6_CCTACCAAGCGATGAC    WT
12_70982708    hf1.d6_GAGTCCGAGACGACGT    Mut
12_70982708    hf1.d6_GGTGCGTAGCCCGAAA    Mut
12_70982708    hf1.d3_CTCGTCAGTTCCATGA    Mut
12_70983413    hf1.d3_ATCGAGTAGCACAGGT    WT
12_70983413    hf1.d3_GATCGTAAGCAATATG    WT
12_70983413    hf1.d3_CTCGTCACACGAGAGT    WT
12_70983413    hf1.d3_TACTTACCAGACACTT    WT
'''

# Read the data into a DataFrame
df = pd.read_csv(StringIO(data), sep='\s+', header=None, names=['Location', 'Cell', 'Status'])

import numpy as np
# Pivot the DataFrame to get the desired matrix format, Na values will be filled with NoCall
matrix = df.pivot_table(index='Location', columns='Cell', values='Status', aggfunc=lambda x: 'Mut' if 'Mut' in x.values else 'WT').fillna('NoCall')

# Sort the columns of the matrix
matrix = matrix.reindex(sorted(matrix.columns), axis=1)

matrix


Cell,hf1.d0_AAATGCCGTCAAACTC,hf1.d0_AGGCCACCATTAGGCT,hf1.d0_CATATGGGTATATGGA,hf1.d0_TTCGGTCGTGGTACAG,hf1.d3_ATCGAGTAGCACAGGT,hf1.d3_CTCGTCACACGAGAGT,hf1.d3_CTCGTCAGTTCCATGA,hf1.d3_GATCGTAAGCAATATG,hf1.d3_TACTTACCAGACACTT,hf1.d6_ACGGGCTGTACCTACA,hf1.d6_CCTACCAAGCGATGAC,hf1.d6_CGCTATCTCGAATGCT,hf1.d6_CTGATCCAGGTGGGTT,hf1.d6_GAGTCCGAGACGACGT,hf1.d6_GGTGCGTAGCCCGAAA,hf1.d6_GTACTCCTCAGCGACC,hf1.d6_TGCGGGTGTCCCGACA
Location,,,,,,,,,,,,,,,,,
12_70902150,WT,NoCall,Mut,WT,NoCall,NoCall,NoCall,NoCall,NoCall,NoCall,NoCall,NoCall,NoCall,NoCall,NoCall,NoCall,NoCall
12_70949151,NoCall,WT,NoCall,NoCall,NoCall,NoCall,NoCall,NoCall,NoCall,Mut,NoCall,WT,WT,NoCall,NoCall,WT,WT
12_70982708,NoCall,NoCall,NoCall,NoCall,NoCall,NoCall,Mut,NoCall,NoCall,NoCall,WT,NoCall,NoCall,Mut,Mut,NoCall,NoCall
12_70983413,NoCall,NoCall,NoCall,NoCall,WT,WT,NoCall,WT,WT,NoCall,NoCall,NoCall,NoCall,NoCall,NoCall,NoCall,NoCall
